# Module 2.3 — Data Prep for Encoder Fine-Tuning
**DeskMate SLM Curriculum · Phase 2 · Notebook 10**

Run on **Google Colab** or **Kaggle** (CPU is fine — no GPU needed for data prep).

By the end you will have:
- A tokenized `DatasetDict` with `input_ids`, `attention_mask`, `intent_label`, `priority_label`
- Label maps saved to `data/processed/label_maps.json`
- A confirmed DataLoader batch with correct shapes
- Dynamic padding working via `DataCollatorWithPadding`

Read `2.3_encoder_dataprep.md` for full theory.

---


## Step 0 — Install & Seed

In [ ]:
%%capture
!pip install -q transformers==4.44.0 datasets==2.21.0 torch==2.3.1


In [ ]:
import random, json, pathlib
import torch
import numpy as np
from datasets import DatasetDict, Dataset, load_from_disk
from transformers import AutoTokenizer, DataCollatorWithPadding
from torch.utils.data import DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"torch       : {torch.__version__}")
import transformers, datasets
print(f"transformers: {transformers.__version__}")
print(f"datasets    : {datasets.__version__}")


## Step 1 — Runtime & Paths

In [ ]:
try:
    import google.colab; RUNTIME = "colab"
    PROJECT_ROOT = pathlib.Path("/content/slm")
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = "kaggle"
        PROJECT_ROOT = pathlib.Path("/kaggle/working/slm")
    except ImportError:
        RUNTIME = "local"
        PROJECT_ROOT = pathlib.Path(".").resolve()

DATA_PROC = PROJECT_ROOT / "data" / "processed"
DATA_PROC.mkdir(parents=True, exist_ok=True)

print(f"Runtime  : {RUNTIME}")
print(f"Data dir : {DATA_PROC}")


## Step 2 — Label Maps

In [ ]:
INTENTS = [
    "account_access", "account_settings", "billing_dispute", "billing_inquiry",
    "cancellation", "data_privacy", "feature_request", "onboarding",
    "outage_report", "payment_failure", "performance_issue", "refund_request",
    "technical_bug", "usage_question", "out_of_scope",
]
PRIORITIES = ["low", "medium", "high"]

INTENT2ID   = {intent: i for i, intent in enumerate(INTENTS)}
ID2INTENT   = {i: intent for intent, i in INTENT2ID.items()}
PRIORITY2ID = {p: i for i, p in enumerate(PRIORITIES)}
ID2PRIORITY = {i: p for p, i in PRIORITY2ID.items()}

print(f"Intent classes  : {len(INTENT2ID)}")
print(f"Priority classes: {len(PRIORITY2ID)}")
print()
for intent, idx in INTENT2ID.items():
    print(f"  {idx:>2}  {intent}")


In [ ]:
# Round-trip verification
for intent in INTENTS:
    assert ID2INTENT[INTENT2ID[intent]] == intent, f"round-trip failed for {intent}"
for priority in PRIORITIES:
    assert ID2PRIORITY[PRIORITY2ID[priority]] == priority

# Save label maps
label_maps = {
    "INTENT2ID": INTENT2ID, "ID2INTENT": {str(k): v for k, v in ID2INTENT.items()},
    "PRIORITY2ID": PRIORITY2ID, "ID2PRIORITY": {str(k): v for k, v in ID2PRIORITY.items()},
}
maps_path = DATA_PROC / "label_maps.json"
maps_path.write_text(json.dumps(label_maps, indent=2))
print(f"Label maps saved: {maps_path}")


## Step 3 — Load Raw Splits

In [ ]:
import pandas as pd

def load_split(name):
    path = DATA_PROC / f"{name}.jsonl"
    if not path.exists():
        # Fallback: create a minimal placeholder so notebook runs standalone
        print(f"  WARNING: {path} not found — creating placeholder (run Modules 2.1-2.2 first)")
        rows = [
            {"text": "I cannot log in to my account.", "intent": "account_access",
             "category": "account", "priority": "medium", "source": "placeholder"},
            {"text": "I was charged twice for the Pro plan.", "intent": "billing_dispute",
             "category": "billing", "priority": "high", "source": "placeholder"},
            {"text": "How do I export my data?", "intent": "usage_question",
             "category": "product", "priority": "low", "source": "placeholder"},
        ] * 5
        path.write_text("\n".join(json.dumps(r) for r in rows))
    return Dataset.from_pandas(pd.read_json(path, lines=True))

aug_path = DATA_PROC / "train_augmented.jsonl"
train_src = "train_augmented" if aug_path.exists() else "train"
print(f"Using train source: {train_src}.jsonl")

raw_ds = DatasetDict({
    "train": load_split(train_src),
    "val":   load_split("val"),
    "test":  load_split("test"),
})

for split, ds in raw_ds.items():
    print(f"  {split:<6}: {len(ds):>6} examples  columns: {ds.column_names}")


In [ ]:
# Inspect one example
print("Sample training example:")
ex = raw_ds["train"][0]
for k, v in ex.items():
    val_str = str(v)[:80] + ("..." if len(str(v)) > 80 else "")
    print(f"  {k:<15}: {val_str}")


In [ ]:
import matplotlib.pyplot as plt

# Text length distribution
lengths = [len(t.split()) for t in raw_ds["train"]["text"]]
plt.figure(figsize=(9, 3))
plt.hist(lengths, bins=40, color="#4C72B0", edgecolor="white")
plt.axvline(128, color="red", linestyle="--", label="max_length=128 tokens (approx)")
plt.xlabel("Word count"); plt.ylabel("Count")
plt.title(f"Ticket length distribution (train, n={len(lengths):,})")
plt.legend(); plt.tight_layout(); plt.show()
print(f"Median length: {int(np.median(lengths))} words")
print(f"p95 length   : {int(np.percentile(lengths, 95))} words")
print(f">128 words   : {sum(l>128 for l in lengths)} ({sum(l>128 for l in lengths)/len(lengths)*100:.1f}%) -- these will be truncated")


## Step 4 — Load Tokenizer

We use MiniLM — fast, 22M params, strong classification performance.
The tokenizer must come from the same checkpoint as the model (Module 2.4).


In [ ]:
MODEL_NAME = "microsoft/MiniLM-L12-H384-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Model        : {MODEL_NAME}")
print(f"Vocab size   : {tokenizer.vocab_size:,}")
print(f"Model max len: {tokenizer.model_max_length}")
print(f"CLS token    : {tokenizer.cls_token!r}  (id {tokenizer.cls_token_id})")
print(f"SEP token    : {tokenizer.sep_token!r}  (id {tokenizer.sep_token_id})")
print(f"PAD token    : {tokenizer.pad_token!r}  (id {tokenizer.pad_token_id})")


In [ ]:
# Visualise one tokenization
sample_text = "I can't log in to my account. Password reset email never arrived."
enc = tokenizer(sample_text, truncation=True, max_length=128)
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])

print(f"Text   : {sample_text!r}")
print(f"IDs    : {enc['input_ids']}")
print(f"Tokens : {tokens}")
print(f"Mask   : {enc['attention_mask']}")
print(f"Length : {len(enc['input_ids'])} tokens")


## Step 5 — Tokenize the Dataset

`padding=False` here — `DataCollatorWithPadding` handles per-batch dynamic padding.
We add `intent_label` and `priority_label` integer columns.


In [ ]:
MAX_LENGTH = 128

def tokenize_fn(examples):
    out = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,    # dynamic padding at batch time
    )
    out["intent_label"]   = [INTENT2ID.get(i, 0)   for i in examples["intent"]]
    out["priority_label"] = [PRIORITY2ID.get(p, 0) for p in examples["priority"]]
    return out

tokenized = raw_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text", "intent", "category", "priority", "source",
                    "fields"],   # drop raw columns
    desc="Tokenizing",
)

print("Tokenized dataset:")
for split, ds in tokenized.items():
    print(f"  {split:<6}: {len(ds):>6} examples  columns: {ds.column_names}")


In [ ]:
# Inspect a tokenized example
print("Tokenized train[0]:")
ex = tokenized["train"][0]
for k, v in ex.items():
    val = v[:10] if isinstance(v, list) and len(v) > 10 else v
    print(f"  {k:<20}: {val}{'...' if isinstance(v, list) and len(v) > 10 else ''}")
print()
print(f"  input_ids length  : {len(tokenized['train'][0]['input_ids'])}")
print(f"  intent_label      : {ID2INTENT[tokenized['train'][0]['intent_label']]}")
print(f"  priority_label    : {ID2PRIORITY[tokenized['train'][0]['priority_label']]}")


## Step 6 — Set Format & DataCollator

`set_format("torch")` makes the dataset return PyTorch tensors instead of Python lists.
`DataCollatorWithPadding` pads each batch to the longest sequence in that batch.


In [ ]:
tokenized.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "intent_label", "priority_label"],
)
print("Format set to torch.")
print(f"Type check: {type(tokenized['train'][0]['input_ids'])}")

# DataCollator
# Rename intent_label → labels so Trainer picks it up automatically for the main task
# We keep priority_label for multi-head use in Module 2.4
collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(f"DataCollatorWithPadding ready (pad_token_id={tokenizer.pad_token_id})")


In [ ]:
# Show dynamic padding in action
sample_batch_raw = [tokenized["train"][i] for i in range(4)]
# Before collation: sequences have different lengths
raw_lens = [len(ex["input_ids"]) for ex in sample_batch_raw]
print(f"Sequence lengths before collation: {raw_lens}")

# Rename for collator compatibility
sample_batch_renamed = []
for ex in sample_batch_raw:
    sample_batch_renamed.append({
        "input_ids":      ex["input_ids"],
        "attention_mask": ex["attention_mask"],
        "labels":         ex["intent_label"],
    })

collated = collator(sample_batch_renamed)
print(f"input_ids shape after collation : {collated['input_ids'].shape}")
print(f"attention_mask shape            : {collated['attention_mask'].shape}")
print(f"labels shape                    : {collated['labels'].shape}")
print()
print("Attention mask for first example (1=real token, 0=pad):")
print(collated["attention_mask"][0].tolist())


## Step 7 — DataLoader Smoke Test

In [ ]:
# Build a DataLoader-compatible dataset with labels renamed
from datasets import Dataset as HFDataset

def rename_labels(examples):
    examples["labels"] = examples["intent_label"]
    return examples

dl_ds = tokenized.map(rename_labels, batched=True)
dl_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels", "priority_label"])

loader = DataLoader(dl_ds["train"], batch_size=16, collate_fn=collator, shuffle=True)

batch = next(iter(loader))
print("DataLoader batch shapes:")
for k, v in batch.items():
    print(f"  {k:<20}: {tuple(v.shape)}  dtype={v.dtype}")
print()
print("Label distribution in this batch:")
from collections import Counter
c = Counter(batch["labels"].tolist())
for label_id, cnt in sorted(c.items()):
    print(f"  {label_id:>2}  {ID2INTENT[label_id]:<25} x{cnt}")


## Step 8 — Save Tokenized Dataset

In [ ]:
save_path = DATA_PROC / "tokenized"
tokenized.save_to_disk(str(save_path))
print(f"Tokenized dataset saved: {save_path}")

# Verify reload
reloaded = load_from_disk(str(save_path))
print(f"Reload check: {list(reloaded.keys())} — {len(reloaded['train'])} train examples")
assert len(reloaded["train"]) == len(tokenized["train"])
print("Reload verified.")


## Checkpoint

> **What does the attention mask do for padded sequences?**

Write your answer below. Strong answer covers:
1. What value the mask has for real tokens vs pad tokens.
2. How the mask is applied in scaled dot-product attention (hint: `-inf` before softmax).
3. Why this matters for the CLS token's representation.


In [ ]:
answer = """
[Write your answer here]
"""
print(answer)


## Deliverable Summary

| Artifact | Location |
|---|---|
| Label maps | `data/processed/label_maps.json` |
| Tokenized dataset | `data/processed/tokenized/` |

**What you've built:** a clean, reloadable tokenized `DatasetDict` with integer labels
and attention masks, ready to drop into `Trainer` in Module 2.4.

**Next:** Module 2.4 — Fine-tune the encoder for intent + priority.
Load `AutoModelForSequenceClassification`, wire the tokenized dataset and collator
into a `Trainer`, and train a classifier that runs in milliseconds on CPU.
